# FinGPT × MedicalGPT：多任务（SFT + DPO）训练流程

目标：把 **FinGPT Benchmark 的多任务金融指令数据（内容）** 映射到 **MedicalGPT 的 SFT + DPO 多阶段训练流程（方法）**。

本 Notebook 会：
- 批量下载你表里的 FinGPT 任务数据集
- 统一转换为 MedicalGPT 支持的 SFT（ShareGPT conversations）与 DPO（pairwise）格式
- 合并通用 SFT + 多任务金融 SFT，合并多任务金融 DPO
- 给出 Qwen2.5-7B 的 SFT / DPO 训练命令（LoRA）

数据格式对齐参考：`docs/datasets.md`
FinGPT 任务与数据集来源参考：[FinGPT 仓库](https://github.com/AI4Finance-Foundation/FinGPT)


In [ ]:
# 0) 配置
from pathlib import Path

BASE_MODEL = "/root/autodl-pub/models/qwen/Qwen2.5-7B-Instruct" # 需要提前下好
TEMPLATE_NAME = "qwen"

# 通用 SFT（MedicalGPT 内置可复用）
GENERAL_SFT_FILES = [
    Path("data/finetune/sharegpt_zh_1K_format.jsonl"),
]

# FinGPT 领域多任务数据集（按你表）
FIN_DATASETS = [
    "FinGPT/fingpt-sentiment-train",
    "FinGPT/fingpt-finred",
    "FinGPT/fingpt-headline",
    "FinGPT/fingpt-ner",
    "FinGPT/fingpt-fiqa_qa",
    "FinGPT/fingpt-fineval",
]
FIN_SPLIT_TRAIN = "train"

OUT_DIR = Path("data/fingpt_medicalgpt_multitask")
RAW_DIR = OUT_DIR / "raw"
FIN_SFT_DIR = OUT_DIR / "fin_sft"
FIN_DPO_DIR = OUT_DIR / "fin_dpo"
MIXED_SFT_DIR = OUT_DIR / "mixed_sft"
MERGED_DPO_DIR = OUT_DIR / "merged_dpo"

MIXED_SFT_FILE = MIXED_SFT_DIR / "train_mixed_sft.jsonl"
MERGED_DPO_FILE = MERGED_DPO_DIR / "train_merged_dpo.jsonl"

SFT_OUT = Path("outputs/fingpt_multitask_sft_lora")
SFT_MERGED_OUT = Path("outputs/fingpt_multitask_sft_merged")
DPO_OUT = Path("outputs/fingpt_multitask_dpo_lora")
DPO_MERGED_OUT = Path("outputs/fingpt_multitask_dpo_merged")

for d in [RAW_DIR, FIN_SFT_DIR, FIN_DPO_DIR, MIXED_SFT_DIR, MERGED_DPO_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("BASE_MODEL:", BASE_MODEL)
print("TEMPLATE_NAME:", TEMPLATE_NAME)
print("GENERAL_SFT_FILES:", GENERAL_SFT_FILES)
print("FIN_DATASETS:", FIN_DATASETS)
print("OUT_DIR:", OUT_DIR)


In [ ]:
# 1) 下载 FinGPT 数据并转换为 MedicalGPT SFT / DPO 格式
import json
import subprocess
from datasets import load_dataset


def safe_name(hf_dataset_name: str) -> str:
    return hf_dataset_name.split("/")[-1]

fin_sft_files = []
fin_dpo_files = []

for ds_name in FIN_DATASETS:
    tag = safe_name(ds_name)
    raw_file = RAW_DIR / f"{tag}_{FIN_SPLIT_TRAIN}.jsonl"
    sft_file = FIN_SFT_DIR / f"{tag}_{FIN_SPLIT_TRAIN}_sharegpt.jsonl"
    dpo_file = FIN_DPO_DIR / f"{tag}_{FIN_SPLIT_TRAIN}_dpo.jsonl"

    ds = load_dataset(ds_name, split=FIN_SPLIT_TRAIN)
    with raw_file.open("w", encoding="utf-8") as f:
        for row in ds:
            f.write(json.dumps(dict(row), ensure_ascii=False) + "\n")

    subprocess.run([
        "python", "fin_to_sharegpt.py",
        "--source_file", str(raw_file),
        "--output_file", str(sft_file),
    ], check=True)

    subprocess.run([
        "python", "fin_to_dpo_pairs.py",
        "--source_file", str(raw_file),
        "--output_file", str(dpo_file),
        "--seed", "42",
    ], check=True)

    fin_sft_files.append(sft_file)
    fin_dpo_files.append(dpo_file)
    print(f"[{ds_name}] raw={raw_file} sft={sft_file} dpo={dpo_file}")


In [ ]:
# 2) 合并训练数据：mixed SFT（通用+金融多任务）与 merged DPO（金融多任务）
from itertools import islice

# 2.1 mixed SFT
with MIXED_SFT_FILE.open("w", encoding="utf-8") as wf:
    for gfile in GENERAL_SFT_FILES:
        if not gfile.exists():
            print(f"[warn] missing general sft file: {gfile}")
            continue
        with gfile.open("r", encoding="utf-8") as rf:
            for line in rf:
                wf.write(line)
    for ffile in fin_sft_files:
        with ffile.open("r", encoding="utf-8") as rf:
            for line in rf:
                wf.write(line)

# 2.2 merged DPO
with MERGED_DPO_FILE.open("w", encoding="utf-8") as wf:
    for ffile in fin_dpo_files:
        with ffile.open("r", encoding="utf-8") as rf:
            for line in rf:
                wf.write(line)

print("mixed sft:", MIXED_SFT_FILE)
print("merged dpo:", MERGED_DPO_FILE)

print("\n[SFT sample]")
with MIXED_SFT_FILE.open("r", encoding="utf-8") as f:
    for line in islice(f, 2):
        print(line.strip())

print("\n[DPO sample]")
with MERGED_DPO_FILE.open("r", encoding="utf-8") as f:
    for line in islice(f, 2):
        print(line.strip())


In [ ]:
# 3) SFT 训练（LoRA）
sft_cmd = [
    "python", "supervised_finetuning.py",
    "--model_name_or_path", BASE_MODEL,
    "--tokenizer_name_or_path", BASE_MODEL,
    "--template_name", TEMPLATE_NAME,
    "--train_file_dir", str(MIXED_SFT_DIR),
    "--validation_split_percentage", "1",
    "--do_train",
    "--use_peft",
    "--num_train_epochs", "3",
    "--per_device_train_batch_size", "2",
    "--gradient_accumulation_steps", "8",
    "--learning_rate", "2e-4",
    "--logging_steps", "10",
    "--save_steps", "200",
    "--model_max_length", "1024",
    "--target_modules", "all",
    "--lora_rank", "8",
    "--lora_alpha", "16",
    "--lora_dropout", "0.05",
    "--torch_dtype", "float16",
    "--device_map", "auto",
    "--output_dir", str(SFT_OUT),
]
print(" ".join(sft_cmd))
# subprocess.run(sft_cmd, check=True)


In [ ]:
# 4) merge SFT LoRA -> 完整模型目录（DPO 起点）
merge_sft_cmd = [
    "python", "merge_peft_adapter.py",
    "--base_model", BASE_MODEL,
    "--tokenizer_path", BASE_MODEL,
    "--lora_model", str(SFT_OUT),
    "--output_dir", str(SFT_MERGED_OUT),
]
print(" ".join(merge_sft_cmd))
# subprocess.run(merge_sft_cmd, check=True)


In [ ]:
# 5) DPO 训练（LoRA）
dpo_cmd = [
    "python", "dpo_training.py",
    "--model_name_or_path", str(SFT_MERGED_OUT),
    "--tokenizer_name_or_path", BASE_MODEL,
    "--template_name", TEMPLATE_NAME,
    "--train_file_dir", str(MERGED_DPO_DIR),
    "--validation_split_percentage", "1",
    "--do_train",
    "--use_peft", "True",
    "--per_device_train_batch_size", "2",
    "--gradient_accumulation_steps", "8",
    "--learning_rate", "5e-7",
    "--max_steps", "200",
    "--max_source_length", "512",
    "--max_target_length", "512",
    "--logging_steps", "10",
    "--save_steps", "200",
    "--eval_steps", "200",
    "--target_modules", "all",
    "--lora_rank", "8",
    "--lora_alpha", "16",
    "--lora_dropout", "0.05",
    "--torch_dtype", "float16",
    "--device_map", "auto",
    "--output_dir", str(DPO_OUT),
]
print(" ".join(dpo_cmd))
# subprocess.run(dpo_cmd, check=True)


In [ ]:
# 6) （可选）merge DPO LoRA -> 可直接推理权重
merge_dpo_cmd = [
    "python", "merge_peft_adapter.py",
    "--base_model", BASE_MODEL,
    "--tokenizer_path", BASE_MODEL,
    "--lora_model", str(DPO_OUT),
    "--output_dir", str(DPO_MERGED_OUT),
]
print(" ".join(merge_dpo_cmd))
